In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ============================================================
# 1. CONFIGURATION
# ============================================================

storage_account = "stsupplychain2026xx"

bronze_events_path = (
    f"abfss://logistics@{storage_account}.dfs.core.windows.net/"
    "bronze/delivery_events/"
)

silver_events_path = (
    f"abfss://logistics@{storage_account}.dfs.core.windows.net/"
    "silver/delivery_events/"
)

silver_shipments_path = (
    f"abfss://logistics@{storage_account}.dfs.core.windows.net/"
    "silver/shipments/"
)

dq_invalid_events_path = (
    f"abfss://logistics@{storage_account}.dfs.core.windows.net/"
    "quality/dq_invalid_records/delivery_events/"
)

dq_metrics_events_path = (
    f"abfss://logistics@{storage_account}.dfs.core.windows.net/"
    "quality/dq_metrics/delivery_events/"
)

# ============================================================
# 2. LECTURE
# ============================================================

bronze_events = (
    spark.read
    .format("delta")
    .load(bronze_events_path)
)

silver_shipments = (
    spark.read
    .format("delta")
    .load(silver_shipments_path)
)

print(
    "Nombre d'événements Bronze :",
    bronze_events.count()
)

# ============================================================
# 3. NORMALISATION
# ============================================================

normalized_events = (
    bronze_events

    .withColumn(
        "event_id",
        F.trim(F.col("event_id"))
    )

    .withColumn(
        "shipment_id",
        F.trim(F.col("shipment_id"))
    )

    .withColumn(
        "event_status",
        F.upper(
            F.trim(
                F.col("event_status")
            )
        )
    )

    .withColumn(
        "location_city",
        F.initcap(
            F.trim(
                F.col("location_city")
            )
        )
    )

    .withColumn(
        "location_country",
        F.initcap(
            F.trim(
                F.col("location_country")
            )
        )
    )

    .withColumn(
        "source_system",
        F.lower(
            F.trim(
                F.col("source_system")
            )
        )
    )
)

# ============================================================
# 4. DÉDUPLICATION PAR event_id
# ============================================================

dedup_window = (
    Window
    .partitionBy("event_id")
    .orderBy(
        F.col("updated_at").desc_nulls_last(),
        F.col("ingestion_timestamp").desc_nulls_last()
    )
)

events_ranked = (
    normalized_events
    .withColumn(
        "_row_number",
        F.row_number().over(dedup_window)
    )
)

duplicate_events = (
    events_ranked
    .filter(
        F.col("_row_number") > 1
    )
)

deduplicated_events = (
    events_ranked
    .filter(
        F.col("_row_number") == 1
    )
    .drop("_row_number")
)

# ============================================================
# 5. PRÉPARER LES SHIPMENTS DE RÉFÉRENCE
# ============================================================

shipment_references = (
    silver_shipments
    .select(
        "shipment_id",
        "shipment_time"
    )
    .dropDuplicates(["shipment_id"])
    .withColumn(
        "_shipment_exists",
        F.lit(True)
    )
)

# ============================================================
# 6. JOIN AVEC SILVER SHIPMENTS
# ============================================================

events_with_refs = (
    deduplicated_events
    .join(
        shipment_references,
        on="shipment_id",
        how="left"
    )
)

# ============================================================
# 7. STATUTS AUTORISÉS
# ============================================================

allowed_statuses = [
    "CREATED",
    "PACKED",
    "SHIPPED",
    "HUB_SCAN",
    "OUT_FOR_DELIVERY",
    "DELIVERED"
]

# ============================================================
# 8. RÈGLES DATA QUALITY
# ============================================================

validated_events = (
    events_with_refs

    .withColumn(
        "dq_event_id_null",
        F.col("event_id").isNull()
        | (F.col("event_id") == "")
    )

    .withColumn(
        "dq_shipment_id_null",
        F.col("shipment_id").isNull()
        | (F.col("shipment_id") == "")
    )

    .withColumn(
        "dq_shipment_not_found",
        F.col("_shipment_exists").isNull()
    )

    .withColumn(
        "dq_event_time_null",
        F.col("event_time").isNull()
    )

    .withColumn(
        "dq_event_before_shipment",
        (
            F.col("event_time").isNotNull()
            & F.col("shipment_time").isNotNull()
            & (
                F.col("event_time")
                < F.col("shipment_time")
            )
        )
    )

    .withColumn(
        "dq_invalid_status",
        F.col("event_status").isNull()
        | (
            ~F.col("event_status")
            .isin(allowed_statuses)
        )
    )

    .withColumn(
        "dq_source_system_null",
        F.col("source_system").isNull()
        | (F.col("source_system") == "")
    )
)

# ============================================================
# 9. CONSTRUIRE dq_errors
# ============================================================

validated_events = (
    validated_events

    .withColumn(
        "dq_errors",
        F.array_compact(
            F.array(

                F.when(
                    F.col("dq_event_id_null"),
                    F.lit("EVENT_ID_NULL")
                ),

                F.when(
                    F.col("dq_shipment_id_null"),
                    F.lit("SHIPMENT_ID_NULL")
                ),

                F.when(
                    F.col("dq_shipment_not_found"),
                    F.lit("SHIPMENT_NOT_FOUND")
                ),

                F.when(
                    F.col("dq_event_time_null"),
                    F.lit("EVENT_TIME_NULL")
                ),

                F.when(
                    F.col("dq_event_before_shipment"),
                    F.lit("EVENT_BEFORE_SHIPMENT")
                ),

                F.when(
                    F.col("dq_invalid_status"),
                    F.lit("INVALID_EVENT_STATUS")
                ),

                F.when(
                    F.col("dq_source_system_null"),
                    F.lit("SOURCE_SYSTEM_NULL")
                )
            )
        )
    )

    .withColumn(
        "dq_is_valid",
        F.size(
            F.col("dq_errors")
        ) == 0
    )
)

# ============================================================
# 10. SÉPARER VALIDES / INVALIDES
# ============================================================

valid_events = (
    validated_events
    .filter(
        F.col("dq_is_valid") == True
    )
)

invalid_events = (
    validated_events
    .filter(
        F.col("dq_is_valid") == False
    )
)

# ============================================================
# 11. AJOUTER L'ORDRE MÉTIER DU LIFECYCLE
# ============================================================

silver_events = (
    valid_events

    .withColumn(
        "event_sequence",
        F.when(
            F.col("event_status") == "CREATED",
            1
        )
        .when(
            F.col("event_status") == "PACKED",
            2
        )
        .when(
            F.col("event_status") == "SHIPPED",
            3
        )
        .when(
            F.col("event_status") == "HUB_SCAN",
            4
        )
        .when(
            F.col("event_status") == "OUT_FOR_DELIVERY",
            5
        )
        .when(
            F.col("event_status") == "DELIVERED",
            6
        )
    )

    .drop(
        "_shipment_exists",
        "dq_event_id_null",
        "dq_shipment_id_null",
        "dq_shipment_not_found",
        "dq_event_time_null",
        "dq_event_before_shipment",
        "dq_invalid_status",
        "dq_source_system_null",
        "dq_errors",
        "dq_is_valid"
    )
)

# ============================================================
# 12. ÉCRITURE SILVER
# ============================================================

(
    silver_events
    .write
    .format("delta")
    .mode("overwrite")
    .save(silver_events_path)
)

# ============================================================
# 13. ÉCRITURE QUARANTAINE
# ============================================================

(
    invalid_events
    .write
    .format("delta")
    .mode("overwrite")
    .save(dq_invalid_events_path)
)

# ============================================================
# 14. MÉTRIQUES DATA QUALITY
# ============================================================

records_checked = deduplicated_events.count()

valid_records = silver_events.count()

invalid_records = invalid_events.count()

duplicate_records = duplicate_events.count()

if records_checked > 0:
    dq_pass_rate = (
        valid_records
        / records_checked
    ) * 100
else:
    dq_pass_rate = 0.0

dq_metrics = [
    (
        "delivery_events",
        records_checked,
        valid_records,
        invalid_records,
        duplicate_records,
        float(dq_pass_rate)
    )
]

dq_metrics_schema = """
dataset STRING,
records_checked LONG,
valid_records LONG,
invalid_records LONG,
duplicate_records LONG,
dq_pass_rate DOUBLE
"""

dq_metrics_df = (
    spark.createDataFrame(
        dq_metrics,
        schema=dq_metrics_schema
    )

    .withColumn(
        "created_at",
        F.current_timestamp()
    )
)

(
    dq_metrics_df
    .write
    .format("delta")
    .mode("overwrite")
    .save(dq_metrics_events_path)
)

# ============================================================
# 15. AFFICHAGE DES RÉSULTATS
# ============================================================

print(
    "Bronze events :",
    bronze_events.count()
)

print(
    "Après déduplication :",
    records_checked
)

print(
    "Doublons détectés :",
    duplicate_records
)

print(
    "Events valides :",
    valid_records
)

print(
    "Events invalides :",
    invalid_records
)

print(
    "DQ pass rate :",
    round(dq_pass_rate, 2),
    "%"
)

print("\n=== SILVER DELIVERY EVENTS ===")

display(
    silver_events
    .select(
        "event_id",
        "shipment_id",
        "event_time",
        "event_status",
        "event_sequence",
        "location_city",
        "location_country",
        "source_system",
        "batch_id"
    )
    .orderBy(
        "shipment_id",
        "event_time"
    )
)

print("\n=== INVALID EVENTS ===")

display(
    invalid_events
    .select(
        "event_id",
        "shipment_id",
        "event_time",
        "event_status",
        "dq_errors",
        "source_file",
        "batch_id"
    )
)

print("\n=== DATA QUALITY METRICS ===")

display(
    dq_metrics_df
)
display(
    invalid_events
    .groupBy("dq_errors")
    .count()
    .orderBy(F.desc("count"))
)



In [0]:
from pyspark.sql import functions as F

display(
    invalid_events
    .filter(
        F.col("dq_event_before_shipment") == True
    )
    .select(
        "event_id",
        "shipment_id",
        "event_status",
        "event_time",
        "shipment_time",
        "updated_at",
        "batch_id",
        "dq_errors"
    )
)
display(
    bronze_events
    .filter(
        F.col("event_id") == "E000050"
    )
    .select(
        "event_id",
        "shipment_id",
        "event_status",
        "event_time",
        "updated_at",
        "batch_id",
        "ingestion_timestamp"
    )
    .orderBy("updated_at")
)